# 实践 14.2  标准化加权指数与多世代遗传响应

《统计遗传与智能育种学》第十四章 · 电子资源 `14b_simer_selection.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：14b_simer_selection.ipynb。实践生成两个正遗传相关的数量性状，固定多性状示例权重[0.65,0.35]，每次重复从同一初始群体出发比较不选择、每代选前20%和选前10%三种情景。每种情景做20次重复，连续跟踪6个世代的真实育种值改变、观察杂合度和次要等位基因频率（MAF）低于0.05的标记比例。Python部分显式声明是独立位点随机交配近似，Notebook中另保留当前SIMER的param.simer()/simer()迁移入口。

**SIMER的R示例（500为初始群体规模）**（外部工具代码模板，本 notebook 不执行）

```r
library(simer)
SP <- param.simer(pop.ind = 500, pop.gen = 6,
                  sel.multi = "index", reprod.way = "randmate")
SP <- simer(SP)
```

## 输入、完整代码、实际输出与解释

输入与可复现条件：SEED=1402；20次重复；每个群体500个个体、180个标记、2个性状、6个世代；不选择/前20%/前10%三种情景；无外部文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED = 1402
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('numpy', np.__version__, 'pandas', pd.__version__, 'scikit-learn', sklearn.__version__)
print('SEED', SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1402


In [2]:
n, m, generations, nrep = 500, 180, 6, 20
master = np.random.default_rng(SEED)
p = master.uniform(0.15, 0.50, m)
B = master.normal(0, 0.11, (m, 2))
B[:, 1] = 0.45 * B[:, 0] + master.normal(0, 0.085, m)
weights = np.array([0.65, 0.35])
resid_sd = np.array([0.9, 0.9])

def run_scenario(base_X, frac, seed, rep, scenario):
    rng = np.random.default_rng(seed)
    X = base_X.copy()
    base_tbv = X @ B
    base_mean = base_tbv.mean(axis=0)
    base_sd = base_tbv.std(axis=0)
    rows = []
    for gen in range(generations):
        tbv = X @ B
        af = X.mean(axis=0) / 2.0
        maf = np.minimum(af, 1.0 - af)
        gain = (tbv.mean(axis=0) - base_mean) / base_sd
        rows.append((scenario, rep, gen, gain[0], gain[1],
                     np.mean(X == 1), np.mean(maf < 0.05)))
        if gen == generations - 1:
            break
        if frac is None:
            parents = np.arange(n)
        else:
            pheno = tbv + rng.normal(0, resid_sd, tbv.shape)
            z = (pheno - pheno.mean(axis=0)) / pheno.std(axis=0)
            score = z @ weights
            nsel = max(2, int(round(frac * n)))
            parents = np.argsort(score)[-nsel:]
        p_parent = X[parents].mean(axis=0) / 2.0
        # 独立位点随机交配近似：不模拟连锁/重组，只传递选择后等位基因频率
        X = rng.binomial(2, p_parent, (n, m)).astype(float)
    return rows

all_rows = []
for rep in range(nrep):
    base_rng = np.random.default_rng(SEED + 1000 + rep)
    base_X = base_rng.binomial(2, p, (n, m)).astype(float)
    all_rows += run_scenario(base_X, None, SEED + 2000 + rep, rep, 'no_selection')
    all_rows += run_scenario(base_X, 0.20, SEED + 3000 + rep, rep, 'select_top20')
    all_rows += run_scenario(base_X, 0.10, SEED + 4000 + rep, rep, 'select_top10')

cols = ['scenario','rep','generation','gain_trait1_sd','gain_trait2_sd','heterozygosity','low_MAF_fraction']
res = pd.DataFrame(all_rows, columns=cols)
base_corr = np.corrcoef((base_X @ B).T)[0, 1]
print('replicates/population/markers/generations', nrep, n, m, generations)
print('base genetic correlation', round(float(base_corr), 3), 'index weights', weights)


replicates/population/markers/generations 20 500 180 6
base genetic correlation 0.468 index weights [0.65 0.35]


In [3]:
trajectory = res.groupby(['scenario','generation'])[
    ['gain_trait1_sd','gain_trait2_sd','heterozygosity','low_MAF_fraction']
].mean().reset_index()
print('mean trajectory across 20 replicates')
print(trajectory.round(3).to_string(index=False))

final = res[res.generation == generations - 1]
summary = final.groupby('scenario').agg(
    gain1_mean=('gain_trait1_sd','mean'), gain1_sd=('gain_trait1_sd','std'),
    gain2_mean=('gain_trait2_sd','mean'), gain2_sd=('gain_trait2_sd','std'),
    heterozygosity_mean=('heterozygosity','mean'),
    low_MAF_mean=('low_MAF_fraction','mean')
)
print('final generation summary')
print(summary.round(3).to_string())


mean trajectory across 20 replicates
    scenario  generation  gain_trait1_sd  gain_trait2_sd  heterozygosity  low_MAF_fraction
no_selection           0           0.000           0.000           0.413             0.000
no_selection           1           0.015           0.011           0.413             0.000
no_selection           2           0.017           0.012           0.412             0.000
no_selection           3           0.015           0.009           0.412             0.000
no_selection           4           0.014          -0.001           0.411             0.000
no_selection           5          -0.002          -0.019           0.410             0.000
select_top10           0           0.000           0.000           0.413             0.000
select_top10           1           1.264           0.973           0.406             0.000
select_top10           2           2.560           1.893           0.398             0.004
select_top10           3           3.793           2.

结果解释：遗传值相关0.468取自最后一次重复的初始群体。20次重复中，不选择对照到第5代的两性状平均改变仍接近0（−0.002和−0.019个初始遗传标准差）。前20%选择到第5代的两性状进展为4.957和3.801，杂合度均值由0代的约0.413降至0.378，低MAF位点比例升至0.024；前10%选择的短期进展更强（6.125和4.609），但杂合度进一步降至0.357，低MAF位点比例升至0.051。这些具体数值只属于当前独立位点简化模拟，不可当成真实育种方案的遗传进展预报。

可修改练习：把权重由[0.65,0.35]交换为[0.35,0.65]；再把环境残差标准差由0.9增到1.5，重跑20次重复，观察表型选择效率和多样性代价如何改变。